Partie 1 – Explorer les données 

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

In [11]:
#Chargement des données CSV
df = pd.read_csv("../data/smart_building_raw.csv")
print("Dimensions :", df.shape)

Dimensions : (507, 14)


In [12]:
#Affichage des premières lignes du dataset
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


In [13]:
#Affichage desS dernières lignes du dataset
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


In [14]:
# Point 5 : combien de variables (colonnes) possède le dataset ?
# df.shape[1] correspond au nombre de colonnes
print(f"Nombre de variables : {df.shape[1]}")

Nombre de variables : 14


In [15]:
# Point 6 : identifier les variables numeriques
# On utilise df.dtypes pour voir le type de chaque colonne,
# puis on liste manuellement celles qui sont vraiment numeriques (mesures continues)
print(df.dtypes)


id_mesure               int64
date                   object
batiment               object
type_batiment          object
zone                   object
temperature           float64
humidite              float64
co2                   float64
occupation            float64
consommation_kwh      float64
mode_climatisation     object
etat_systeme           object
jour_semaine           object
alerte                 object
dtype: object


In [16]:
# On definit explicitement la liste des variables numeriques du dataset
variables_numeriques = ["temperature", "humidite", "co2", "occupation", "consommation_kwh"]
print("Variables numeriques :", variables_numeriques)

Variables numeriques : ['temperature', 'humidite', 'co2', 'occupation', 'consommation_kwh']


In [17]:
# Point 7 : identifier les variables categorielles
# Ce sont les colonnes qui representent des categories (texte, classes), pas des mesures continues
variables_categorielles = ["batiment", "type_batiment", "zone", "mode_climatisation", "etat_systeme", "jour_semaine", "alerte"]
print("Variables categorielles :", variables_categorielles)

Variables categorielles : ['batiment', 'type_batiment', 'zone', 'mode_climatisation', 'etat_systeme', 'jour_semaine', 'alerte']


In [18]:
# Point 8 : identifier la ou les variables de type date
variable_date = "date"
print("Variable de type date :", variable_date)
print(df[variable_date].head())

Variable de type date : date
0    2025-02-13 06:00:00
1    2025-03-10 12:00:00
2    2025-05-04 00:00:00
3    2025-01-19 00:00:00
4    2025-04-24 06:00:00
Name: date, dtype: object


In [19]:
# Point 9 : identifier la ou les variables qui servent d'identifiant
variable_id = "id_mesure"
print("Variable identifiant :", variable_id)
print("Nombre de valeurs uniques :", df[variable_id].nunique())
print("Nombre total de lignes    :", df.shape[0])

Variable identifiant : id_mesure
Nombre de valeurs uniques : 500
Nombre total de lignes    : 507


In [20]:
# Point 10 : statistiques descriptives des variables numeriques
# describe() calcule automatiquement : count, mean, std, min, 25%, 50% (mediane), 75%, max
df[variables_numeriques].describe()

,temperature,humidite,co2,occupation,consommation_kwh
count,495.000000,496.000000,500.000000,501.000000,502.000000
mean,24.154141,57.864113,844.150000,44.850299,169.069323
std,7.418465,16.026336,582.181386,24.949139,53.164294
min,-30.000000,-12.000000,89.000000,-20.000000,-100.000000
25%,21.600000,49.275000,623.750000,27.000000,136.875000
50%,24.000000,57.550000,787.500000,46.000000,169.800000
75%,26.000000,65.750000,952.000000,61.000000,202.975000
max,96.000000,160.000000,6000.000000,116.000000,336.200000


## Point 11 — Variables potentiellement problématiques

D'après les statistiques descriptives (point 10) et un premier coup d'œil sur les données :

- **temperature** : le minimum et le maximum semblent très extrêmes pour un bâtiment (ex. valeurs négatives fortes ou supérieures à 70°C) → probablement des erreurs de capteur.
- **humidite** : l'humidité relative doit être comprise entre 0 et 100 % ; or on observe des valeurs négatives et des valeurs bien au-dessus de 100 → incohérent physiquement.
- **co2** : certaines valeurs atteignent plusieurs milliers de ppm, largement au-dessus des niveaux réalistes en intérieur → suspect.
- **consommation_kwh** : présence de valeurs négatives, alors qu'une consommation ne peut pas être négative.
- **occupation** : présence de valeurs négatives, alors qu'un nombre de personnes ne peut pas être négatif.
- **type_batiment**, **mode_climatisation**, **jour_semaine** : catégories mal orthographiées ou avec une casse incohérente (ex. "École" / "ÉCOLE" / "ecole").
- **date** : certaines valeurs ne sont pas des dates valides (ex. "date_invalide", "31/99/2025").

Ces problèmes seront traités un par un dans les points suivants (incohérences, valeurs manquantes, doublons, valeurs aberrantes).

In [21]:
# Point 12a : rechercher les valeurs d'humidite negatives (physiquement impossible)
humidite_negative = df["humidite"] < 0
print("Nombre de valeurs d'humidite negatives :", humidite_negative.sum())
df.loc[humidite_negative, ["id_mesure", "humidite"]]

Nombre de valeurs d'humidite negatives : 3


,id_mesure,humidite
281,1126,-5.0
335,1036,-8.0
366,1216,-12.0


In [22]:
# Point 12b : rechercher les valeurs d'humidite superieures a 100 (physiquement impossible)
humidite_superieure_100 = df["humidite"] > 100
print("Nombre de valeurs d'humidite > 100 :", humidite_superieure_100.sum())
df.loc[humidite_superieure_100, ["id_mesure", "humidite"]]

Nombre de valeurs d'humidite > 100 : 7


,id_mesure,humidite
59,1246,108.0
103,1016,145.0
128,1156,160.0
160,1186,125.0
268,1276,140.0
327,1066,132.0
342,1096,110.0


In [23]:
# Point 12c : rechercher les temperatures extremement elevees (ou basses) pour un batiment
# Seuils physiquement realistes pour l'interieur d'un batiment : entre -10 et 45 degres
temperature_aberrante = (df["temperature"] < -10) | (df["temperature"] > 45)
print("Nombre de temperatures aberrantes :", temperature_aberrante.sum())
df.loc[temperature_aberrante, ["id_mesure", "temperature"]]

Nombre de temperatures aberrantes : 6


,id_mesure,temperature
7,1141,72.5
116,1181,96.0
161,1061,88.0
185,1221,-25.0
358,1101,-30.0
499,1021,95.2


In [24]:
# Point 12d : rechercher les valeurs d'occupation negatives (un nombre de personnes ne peut pas etre negatif)
occupation_negative = df["occupation"] < 0
print("Nombre de valeurs d'occupation negatives :", occupation_negative.sum())
df.loc[occupation_negative, ["id_mesure", "occupation"]]

Nombre de valeurs d'occupation negatives : 5


,id_mesure,occupation
22,1031,-5.0
376,1231,-8.0
430,1081,-12.0
487,1131,-2.0
494,1331,-20.0


In [25]:
# Point 12e : rechercher les valeurs de consommation energetique negatives (impossible physiquement)
consommation_negative = df["consommation_kwh"] < 0
print("Nombre de valeurs de consommation negatives :", consommation_negative.sum())
df.loc[consommation_negative, ["id_mesure", "consommation_kwh"]]

Nombre de valeurs de consommation negatives : 4


,id_mesure,consommation_kwh
59,1246,-15.0
154,1046,-50.0
199,1146,-20.0
441,1346,-100.0


In [26]:
# Point 12f : transformer les valeurs manifestement erronees en valeurs manquantes (NaN)
# On ne peut pas deviner la vraie valeur d'origine, donc on ne "corrige" pas au hasard :
# on marque ces valeurs comme manquantes, elles seront traitees comme telles (imputation) plus tard.

df.loc[humidite_negative, "humidite"] = np.nan
df.loc[humidite_superieure_100, "humidite"] = np.nan
df.loc[temperature_aberrante, "temperature"] = np.nan
df.loc[occupation_negative, "occupation"] = np.nan
df.loc[consommation_negative, "consommation_kwh"] = np.nan

print("Nombre total de valeurs transformees en NaN pour :")
print(" - humidite         :", humidite_negative.sum() + humidite_superieure_100.sum())
print(" - temperature       :", temperature_aberrante.sum())
print(" - occupation        :", occupation_negative.sum())
print(" - consommation_kwh  :", consommation_negative.sum())

Nombre total de valeurs transformees en NaN pour :
 - humidite         : 10
 - temperature       : 6
 - occupation        : 5
 - consommation_kwh  : 4


In [27]:
# Point 12g : rechercher les categories mal orthographiees ou avec une casse incoherente
# On regarde les valeurs uniques de chaque variable categorielle pour reperer les anomalies
for c in ["type_batiment", "mode_climatisation", "jour_semaine"]:
    print(f"--- {c} ---")
    print(df[c].value_counts(dropna=False))
    print()

--- type_batiment ---
type_batiment
Bureau               210
Centre commercial     65
École                 59
Hôpital               58
Entrepôt              52
Université            48
NaN                    4
BUREAU                 2
ecole                  1
ÉCOLE                  1
Bureu                  1
bureau                 1
 UNIVERSITÉ            1
hôpital                1
centre commercial      1
 Bureau                1
entrepot               1
Name: count, dtype: int64

--- mode_climatisation ---
mode_climatisation
Normal     271
Eco        131
Boost       96
NaN          5
normal       1
BOOST        1
normale      1
Normal       1
Name: count, dtype: int64

--- jour_semaine ---
jour_semaine
Vendredi    75
Jeudi       74
Mercredi    73
Samedi      72
Dimanche    70
Lundi       69
Mardi       69
NaN          5
Name: count, dtype: int64



In [28]:
# Point 12h : normaliser les categories textuelles
# Etape 1 : supprimer les espaces en debut/fin de chaine (strip)
for c in variables_categorielles:
    df[c] = df[c].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Etape 2 : uniformiser la casse et corriger les fautes evidentes, categorie par categorie
corrections_type_batiment = {
    "ecole": "École", "école": "École",
    "bureau": "Bureau", "bureu": "Bureau",
    "hôpital": "Hôpital",
    "université": "Université",
    "entrepot": "Entrepôt", "entrepôt": "Entrepôt",
    "centre commercial": "Centre commercial",
}
corrections_mode_clim = {"normal": "Normal", "normale": "Normal", "boost": "Boost", "eco": "Eco"}

df["type_batiment"] = df["type_batiment"].str.lower().map(corrections_type_batiment).fillna(df["type_batiment"])
df["mode_climatisation"] = df["mode_climatisation"].str.lower().map(corrections_mode_clim).fillna(df["mode_climatisation"])

print("--- type_batiment apres nettoyage ---")
print(df["type_batiment"].value_counts(dropna=False))
print()
print("--- mode_climatisation apres nettoyage ---")
print(df["mode_climatisation"].value_counts(dropna=False))

--- type_batiment apres nettoyage ---
type_batiment
Bureau               215
Centre commercial     66
École                 61
Hôpital               59
Entrepôt              53
Université            49
NaN                    4
Name: count, dtype: int64

--- mode_climatisation apres nettoyage ---
mode_climatisation
Normal    274
Eco       131
Boost      97
NaN         5
Name: count, dtype: int64


In [29]:
# Point 13a : calculer le nombre et le pourcentage de valeurs manquantes par colonne
na_count = df.isna().sum()
na_pct = (df.isna().mean() * 100).round(2)
na_summary = pd.DataFrame({"nb_manquantes": na_count, "pct_manquantes": na_pct})
na_summary = na_summary[na_summary["nb_manquantes"] > 0].sort_values("nb_manquantes", ascending=False)
na_summary

,nb_manquantes,pct_manquantes
humidite,21,4.14
temperature,18,3.55
occupation,11,2.17
consommation_kwh,9,1.78
co2,7,1.38
mode_climatisation,5,0.99
jour_semaine,5,0.99
type_batiment,4,0.79


In [30]:
# Point 13b : quelle variable possede le plus de valeurs manquantes ?
variable_plus_manquante = na_summary.index[0]
print(f"Variable avec le plus de valeurs manquantes : {variable_plus_manquante}")
print(f"Nombre : {na_summary.loc[variable_plus_manquante, 'nb_manquantes']}")
print(f"Pourcentage : {na_summary.loc[variable_plus_manquante, 'pct_manquantes']}%")

Variable avec le plus de valeurs manquantes : humidite
Nombre : 21
Pourcentage : 4.14%


c) Quelle stratégie utiliser pour les valeurs manquantes ?
d) Peut-on supprimer toutes les lignes contenant des valeurs manquantes ?
e) Dans quels cas utiliser la moyenne ?
f) Quand préférer la médiane ?
g) Comment traiter une variable catégorielle ?

## Points 13c à 13g — Stratégie pour les valeurs manquantes

**c) Quelle stratégie utiliser ?**
Il n'existe pas une seule bonne réponse : le choix dépend du type de variable (numérique ou catégorielle), de la proportion de valeurs manquantes, et de la distribution des données. Ici, les pourcentages sont faibles (moins de 5% par colonne), donc l'imputation (remplacement par une valeur statistique) est préférable à la suppression.

**d) Peut-on supprimer toutes les lignes avec une valeur manquante ?**
Non, ce n'est pas recommandé ici. Plusieurs colonnes différentes contiennent des `NaN`, souvent sur des lignes différentes : supprimer toute ligne concernée ferait perdre une part importante et non négligeable du dataset (plusieurs dizaines de lignes cumulées), et pourrait introduire un biais si les valeurs manquantes ne sont pas réparties au hasard.

**e) Dans quels cas utiliser la moyenne ?**
La moyenne est adaptée quand la distribution de la variable est à peu près symétrique et sans valeurs extrêmes marquées — elle est alors représentative du "centre" des données.

**f) Quand préférer la médiane ?**
La médiane est préférable dès que la distribution est asymétrique ou contient des valeurs extrêmes (outliers) : contrairement à la moyenne, elle n'est pas influencée par ces valeurs, ce qui la rend plus robuste. C'est le cas ici pour des variables comme `consommation_kwh` ou `co2`.

**g) Comment traiter une variable catégorielle ?**
On ne peut pas calculer de moyenne ou de médiane sur du texte. La stratégie standard est d'imputer par le **mode** (la catégorie la plus fréquente), ou d'introduire une catégorie explicite type `"Inconnu"` si l'on souhaite conserver la trace de l'absence d'information plutôt que de la masquer.

In [31]:
# Point 14a et 14b : identifier et afficher les doublons
# keep=False permet de marquer TOUTES les occurrences d'un doublon (pas seulement la 2eme)
doublons_stricts = df.duplicated(keep=False)
print("Nombre de lignes impliquees dans un doublon strict (toutes colonnes identiques) :", doublons_stricts.sum())
df[doublons_stricts].sort_values("id_mesure")

Nombre de lignes impliquees dans un doublon strict (toutes colonnes identiques) : 14


,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
117,1026,2025-01-07 06:00:00,B6,Bureau,A,23.0,74.3,952.0,89.0,247.6,Normal,Normal,Mardi,Oui
192,1026,2025-01-07 06:00:00,B6,Bureau,A,23.0,74.3,952.0,89.0,247.6,Normal,Normal,Mardi,Oui
9,1118,2025-01-30 06:00:00,B1,Bureau,A,22.0,NaN,479.0,64.0,145.0,Normal,Normal,Jeudi,Non
123,1118,2025-01-30 06:00:00,B1,Bureau,A,22.0,NaN,479.0,64.0,145.0,Normal,Normal,Jeudi,Non
23,1264,2025-03-07 18:00:00,B1,Bureau,B,21.1,38.3,1034.0,74.0,199.2,Normal,Panne,Vendredi,Oui
434,1264,2025-03-07 18:00:00,B1,Bureau,B,21.1,38.3,1034.0,74.0,199.2,Normal,Panne,Vendredi,Oui
455,1320,2025-03-21 18:00:00,B7,Bureau,B,27.5,27.2,359.0,33.0,185.8,Normal,Normal,Vendredi,Non
464,1320,2025-03-21 18:00:00,B7,Bureau,B,27.5,27.2,359.0,33.0,185.8,Normal,Normal,Vendredi,Non
238,1402,2025-04-11 06:00:00,B2,École,A,22.6,44.0,868.0,62.0,172.5,Eco,Normal,Vendredi,Non
418,1402,2025-04-11 06:00:00,B2,École,A,22.6,44.0,868.0,62.0,172.5,Eco,Normal,Vendredi,Non
